# YOLO26 numbered-cube detector fine-tuning

Fine-tune the existing five-shape checkpoint for **one class: cube**. The detection head adapts from five classes to one; the original checkpoint remains unchanged.

The dataset export currently contains **400 matched images** (99 hand-reviewed, 301 AI-reviewed), not 500. Update the export after locating the other 100 image/label pairs. The 80-image validation set is a chronological holdout from the same camera session, so its metrics are provisional. Capture a separate session for the final test.

On Kaggle, enable a GPU and attach the dataset and model at the paths in section 1. Install `ultralytics` if needed. Locally, run this notebook from the project environment with `ultralytics`, `torch`, `opencv-python`, `matplotlib`, `jupyterlab`, and `ipykernel` installed. Training starts in section 4 when you run that cell. CPU training can be slow; a CUDA GPU is preferred.

## 1. Paths and environment

In [ ]:
from pathlib import Path
import csv
import random
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Kaggle inputs are read-only. Training results and the portable YAML go to /kaggle/working.
KAGGLE_DATASET = Path('/kaggle/input/datasets/aymenslimani/cube-fine-tunning/yolo_cube_finetune')
KAGGLE_WEIGHTS = Path('/kaggle/input/models/aymenslimani/yolo-shape-detection/transformers/default/1/robotic_E1_camera_finetune_best.pt')
if KAGGLE_DATASET.is_dir():
    DATASET = KAGGLE_DATASET
    SOURCE_WEIGHTS = KAGGLE_WEIGHTS
    RUNS = Path('/kaggle/working/cube_only_yolo26')
else:
    def find_project_root(start: Path) -> Path:
        for candidate in [start.resolve(), *start.resolve().parents]:
            if (candidate / 'vision_experiments/checkpoints/robotic_E1_camera_finetune_best.pt').is_file():
                return candidate
        raise FileNotFoundError('Project root not found; check the Kaggle input paths or local project folder.')

    ROOT = find_project_root(Path.cwd())
    DATASET = ROOT / 'dataset/cube_only_v1/yolo_cube_finetune'
    SOURCE_WEIGHTS = ROOT / 'vision_experiments/checkpoints/robotic_E1_camera_finetune_best.pt'
    RUNS = ROOT / 'vision_experiments/runs/cube_only_yolo26'

assert DATASET.is_dir(), f'Dataset folder not found: {DATASET}'
assert SOURCE_WEIGHTS.is_file(), f'Model checkpoint not found: {SOURCE_WEIGHTS}'
for split in ('train', 'val'):
    assert (DATASET / 'images' / split).is_dir(), f'Missing images/{split} in {DATASET}'
    assert (DATASET / 'labels' / split).is_dir(), f'Missing labels/{split} in {DATASET}'

# The uploaded data.yaml may contain the Windows path from the export machine.
# Write a Kaggle/local runtime YAML with the correct absolute dataset path.
RUNS.mkdir(parents=True, exist_ok=True)
DATA_YAML = RUNS / 'cube_data.yaml'
DATA_YAML.write_text(
    f'path: "{DATASET.as_posix()}"\n'
    'train: images/train\n'
    'val: images/val\n'
    'nc: 1\n'
    'names: [cube]\n',
    encoding='utf-8',
)
print('Dataset:', DATASET)
print('Runtime YAML:', DATA_YAML)
print('Source checkpoint:', SOURCE_WEIGHTS)
print('Training output:', RUNS)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Confirm checkpoint and dataset

In [ ]:
source_model = YOLO(str(SOURCE_WEIGHTS))
print('Original classes:', source_model.names)
assert source_model.names.get(0) == 'cube'
with (DATASET / 'split_manifest.csv').open(newline='', encoding='utf-8') as stream:
    split_rows = list(csv.DictReader(stream))
for split in ('train', 'val'):
    rows = [row for row in split_rows if row['split'] == split]
    image_paths = list((DATASET / 'images' / split).glob('*.jpg'))
    label_paths = list((DATASET / 'labels' / split).glob('*.txt'))
    assert len(rows) == len(image_paths) == len(label_paths), f'{split}: unmatched files'
    assert {p.stem for p in image_paths} == {p.stem for p in label_paths}
    print(f'{split}: {len(rows)} images, {sum(int(row["cube_count"]) for row in rows)} cubes')
print('Reviewed sources:', {status: sum(row['status'] == status for row in split_rows) for status in ('reviewed', 'ai_reviewed')})

## 3. Preview training labels

Boxes should include each cube body and its visible dark side faces, while excluding the cast shadow.

In [ ]:
import cv2
random.seed(26)
samples = random.sample(sorted((DATASET / 'images/train').glob('*.jpg')), k=min(6, len(list((DATASET / 'images/train').glob('*.jpg')))))
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for axis, image_path in zip(axes.flat, samples):
    frame = cv2.imread(str(image_path))
    height, width = frame.shape[:2]
    label_path = DATASET / 'labels/train' / f'{image_path.stem}.txt'
    for line in label_path.read_text().splitlines():
        _, xc, yc, bw, bh = map(float, line.split())
        x1, y1 = int((xc - bw / 2) * width), int((yc - bh / 2) * height)
        x2, y2 = int((xc + bw / 2) * width), int((yc + bh / 2) * height)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 210, 255), 2)
    axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axis.set_title(image_path.stem[-18:], fontsize=9)
    axis.axis('off')
for axis in list(axes.flat)[len(samples):]:
    axis.axis('off')
plt.tight_layout()
plt.show()

## 4. Fine-tune YOLO26 for cubes

This creates a new run under `/kaggle/working/cube_only_yolo26/` on Kaggle (or the local project runs folder). Keep the source checkpoint for the five-shape experiment. If training is interrupted, resume from that run's `weights/last.pt` rather than starting this cell again.

The augmentation is moderate for cube detection. Digit rotation robustness remains the classifier's separate job.

In [ ]:
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
BATCH = 8 if DEVICE != 'cpu' else 4
EPOCHS = 60
RUN_NAME = 'camera_cubes_v1'
assert not (RUNS / RUN_NAME).exists(), 'Run already exists. Choose a new RUN_NAME or resume its weights/last.pt.'
print(f'Device={DEVICE}, batch={BATCH}, epochs={EPOCHS}')
trained = YOLO(str(SOURCE_WEIGHTS))
train_result = trained.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=DEVICE,
    workers=0,  # reliable with Jupyter on Windows
    project=str(RUNS),
    name=RUN_NAME,
    exist_ok=False,
    pretrained=True,
    patience=15,
    seed=26,
    deterministic=True,
    degrees=30,
    translate=0.10,
    scale=0.30,
    fliplr=0.5,
    flipud=0.5,
    mosaic=0.5,
    close_mosaic=10,
    plots=True,
)
print('Best checkpoint:', RUNS / RUN_NAME / 'weights/best.pt')

## 5. Validate the best checkpoint

These scores use the same-session chronological holdout. Record them for development, then test on a new capture session before choosing a deployment checkpoint.

In [ ]:
BEST = RUNS / RUN_NAME / 'weights/best.pt'
assert BEST.is_file(), f'Missing checkpoint: {BEST}'
cube_model = YOLO(str(BEST))
print('Trained classes:', cube_model.names)
metrics = cube_model.val(data=str(DATA_YAML), split='val', imgsz=640, device=DEVICE, plots=True)
print('Precision:', round(float(metrics.box.mp), 4))
print('Recall:', round(float(metrics.box.mr), 4))
print('mAP50:', round(float(metrics.box.map50), 4))
print('mAP50-95:', round(float(metrics.box.map), 4))

## 6. Inspect validation predictions

Check missed cubes, duplicate detections, cast shadows, and partial cubes visually. The saved `best.pt` is the detector to test next with the live camera and the digit classifier.

In [ ]:
validation_images = sorted((DATASET / 'images/val').glob('*.jpg'))
examples = random.Random(27).sample(validation_images, k=min(6, len(validation_images)))
predictions = cube_model.predict([str(path) for path in examples], imgsz=640, conf=0.25, device=DEVICE, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for axis, result in zip(axes.flat, predictions):
    axis.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    axis.set_title(Path(result.path).stem[-18:], fontsize=9)
    axis.axis('off')
for axis in list(axes.flat)[len(predictions):]:
    axis.axis('off')
plt.tight_layout()
plt.show()